<a href="https://colab.research.google.com/github/babaronlinecom/Projects-Solutions/blob/master/RAG_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -qU google-generativeai langchain-google-genai chromadb pypdf
warnings.filterwarnings("ignore")


In [ ]:
!pip install -U langchain-community -qU

In [ ]:
!pip install langchain_google_genai langchain_chroma -qU

In [ ]:
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')

In [ ]:
!sudo apt -y -qq install tesseract-ocr libtesseract-dev -qU

!sudo apt-get -y -qq install poppler-utils libxml2-dev libxslt1-dev antiword unrtf poppler-utils pstotext tesseract-ocr flac ffmpeg lame libmad0 libsox-fmt-mp3 sox libjpeg-dev swig -qU

!pip install langchain -qU

E: Command line option 'U' [from -qU] is not understood in combination with the other options.
E: Command line option 'U' [from -qU] is not understood in combination with the other options.


In [ ]:
import urllib
import warnings
from pathlib import Path as p
from pprint import pprint
import pandas as pd
from langchain import PromptTemplate
from langchain.chains.question_answering import load_qa_chain
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
warnings.filterwarnings("ignore")

In [ ]:
loader = PyPDFLoader("/content/test.pdf")
data = loader.load()
len(data)

11

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=10000, chunk_overlap=1000)
context = "\n\n".join(str(p.page_content) for p in data)
texts = text_splitter.split_text(context)

print("total number of documents : " ,len(texts))

total number of documents :  5


In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash-exp",google_api_key=GOOGLE_API_KEY,
                             temperature=0.2,convert_system_message_to_human=True)


In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001",google_api_key=GOOGLE_API_KEY)

In [ ]:
vector_index = Chroma.from_texts(texts, embeddings).as_retriever(search_kwargs={"k":5})


In [ ]:
template = """Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer. Keep the answer as concise as possible. Always say "thanks for asking!" at the end of the answer.
{context}
Question: {question}
Helpful Answer:"""


In [ ]:
QA_CHAIN_PROMPT = PromptTemplate.from_template(template)


In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=vector_index,
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
)


In [ ]:
question = "Describe the full potential of LLMs in real-world in detail?"
result = qa_chain({"query": question})
result["result"]

'LLMs have the potential to revolutionize various applications, from intelligent conversational agents to sophisticated data analysis tools. They can understand and generate human-like text, enabling advancements in numerous domains. thanks for asking!\n'